In [1]:
# Set seed
# Save png and pkl in "./results"
# NN_type and INPUT are used only for the output file name
import numpy as np
NN_type = "CNN"
infile = 'multi_well_data.csv'

# mult=np.array([5, 50,1, 5, 0.1, 5]) # MLP
mult=np.array([5, 50,1, 5, 1, 0.1, 5]) # CNN
# mult=np.array([5, 50,1, 5, 0.1, 3]) # RNN

In [2]:

import pandas as pd

import datetime
from matplotlib import pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_squared_error
import random
from numpy.random import seed
from tensorflow import set_random_seed

from utility import *
import importlib
import time
import scipy.spatial as scp
import math
import numpy.matlib
import copy
import pickle
import array

# change table to lag table

In [3]:
def table2lags(table, max_lag, min_lag=0, separator='_'):
    values=[]
    for i in range(min_lag, max_lag + 1):
        values.append(table.shift(i).copy())
        values[-1].columns = [c + separator + str(i) for c in table.columns]
    return pd.concat(values, axis=1)

# Input data

In [4]:
# look_back = 5
look_next = 1


# load data
dataset = pd.read_csv(infile, header=0, index_col=1)
dataset.index = pd.to_datetime(dataset.index)
    
# Add month/day/year
# dataset = dataset.assign(MoY= dataset.index.month)
# dataset = dataset.assign(DoY= dataset.index.dayofyear)
dataset = dataset.assign(WoY= dataset.index.weekofyear)
# dataset = dataset.assign(Y= dataset.index.year)
dataset_head = dataset.columns
dataset = dataset[dataset_head[1:]]

# normalize
max_val = np.max(dataset)
min_val = np.min(dataset)
min_val[0] = 0
min_val[1] = 0
min_val[2] = 0
for i in range(max_val.shape[0]):
    dataset[dataset_head[i+1]] = (dataset[dataset_head[i+1]]-min_val[i])/(max_val[i]-min_val[i])
n_features = dataset.shape[1]

# Split Data

In [5]:
# change table to lag table
def SplitData(look_back,dataset):
#     look_back = 3
    dataset_lag = table2lags(dataset, look_back)
    

    X = dataset_lag[look_back:]
    Y1 = X[X.columns[0]]
    Y2 = X[X.columns[1]]
    Y3 = X[X.columns[2]]
    
    
    

    # training date range
    train_start = dataset.index[0] + datetime.timedelta(days=look_back)
    train_end   = train_start + datetime.timedelta(days=365*3)

    train_X = X.loc[train_start:train_end]
    
    train_Y1 = Y1.loc[train_start+datetime.timedelta(days=look_next):train_end+datetime.timedelta(days=look_next)]
    train_Y2 = Y2.loc[train_start+datetime.timedelta(days=look_next):train_end+datetime.timedelta(days=look_next)]
    train_Y3 = Y3.loc[train_start+datetime.timedelta(days=look_next):train_end+datetime.timedelta(days=look_next)]
    
    train_Y = pd.concat([train_Y1, train_Y2, train_Y3], axis=1)



    # testing date range
    test_start  = train_end + datetime.timedelta(days=1)
    test_end    = dataset.index[-1]

    test_X = X.loc[test_start:test_end-datetime.timedelta(days=look_next)] # will not use last X
    test_Y1 = copy.deepcopy(Y1.loc[test_start+datetime.timedelta(days=look_next):test_end+datetime.timedelta(days=look_next)])
    test_Y2 = copy.deepcopy(Y2.loc[test_start+datetime.timedelta(days=look_next):test_end+datetime.timedelta(days=look_next)])
    test_Y3 = copy.deepcopy(Y3.loc[test_start+datetime.timedelta(days=look_next):test_end+datetime.timedelta(days=look_next)])
    
    test_Y = pd.concat([test_Y1, test_Y2, test_Y3], axis=1)
    
    ## Reshape FOR TRAIN
    train_X_in = np.reshape(train_X.values, (train_X.values.shape[0],look_back+1,n_features))
    train_Y_in = np.reshape(train_Y.values, (train_Y.values.shape[0],3))
    test_X_in = np.reshape(test_X.values, (test_X.values.shape[0],look_back+1,n_features))

#     print("="*70)
#     print("train_start=",train_start)
#     print("train_end=",train_end)
#     print("="*70)
#     print("test_start=",test_start)
#     print("test_end=",test_end)
#     print("="*70)
    
    
    return(train_X_in,train_Y_in,test_X_in,test_Y)

# Set up optimization problem

In [6]:
def myfun():    
    return 0

In [9]:
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_random_'+NN_type+'_random_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = data.best_Y
        tt = data.S[np.argmin(data.Y)]*mult
        latex = [" & " + str(np.round(tt[i],1)) for i in [1,0,2,3,4,5,6]]
        print(latex)
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_GP_'+NN_type+'_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = data.best_Y
#         print(data.xbest*mult)
        tt = data.S[np.argmin(data.Y)]*mult
        latex = [" & " + str(np.round(tt[i],1)) for i in [1,0,2,3,4,5,6]]
        print(latex)
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_RBF_'+NN_type+'_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = data.best_Y
        tt = data.S[np.argmin(data.Y)]*mult
        latex = [" & " + str(np.round(tt[i],1)) for i in [1,0,2,3,4,5,6]]
        print(latex)

[' & 350.0', ' & 155.0', ' & 5.0', ' & 20.0', ' & 5.0', ' & 0.1', ' & 365.0']
[' & 400.0', ' & 195.0', ' & 5.0', ' & 45.0', ' & 4.0', ' & 0.5', ' & 320.0']
[' & 250.0', ' & 80.0', ' & 6.0', ' & 35.0', ' & 4.0', ' & 0.3', ' & 350.0']
[' & 500.0', ' & 175.0', ' & 3.0', ' & 45.0', ' & 5.0', ' & 0.4', ' & 310.0']
[' & 100.0', ' & 95.0', ' & 6.0', ' & 35.0', ' & 3.0', ' & 0.4', ' & 360.0']
[' & 350.0', ' & 155.0', ' & 5.0', ' & 20.0', ' & 3.0', ' & 0.1', ' & 330.0']
[' & 100.0', ' & 90.0', ' & 5.0', ' & 25.0', ' & 2.0', ' & 0.2', ' & 225.0']
[' & 250.0', ' & 80.0', ' & 6.0', ' & 35.0', ' & 4.0', ' & 0.3', ' & 350.0']
[' & 350.0', ' & 155.0', ' & 5.0', ' & 20.0', ' & 3.0', ' & 0.1', ' & 330.0']
[' & 450.0', ' & 185.0', ' & 2.0', ' & 35.0', ' & 5.0', ' & 0.4', ' & 210.0']
[' & 450.0', ' & 50.0', ' & 5.0', ' & 45.0', ' & 2.0', ' & 0.2', ' & 195.0']
[' & 500.0', ' & 170.0', ' & 5.0', ' & 50.0', ' & 2.0', ' & 0.4', ' & 310.0']
[' & 400.0', ' & 190.0', ' & 5.0', ' & 45.0', ' & 3.0', ' & 0.1', ' &

In [25]:
# example data

loss_tb = np.zeros((5,50))
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_random_'+NN_type+'_random_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = np.array(data.best_Y)
        tt = data.S[np.argmin(data.Y)]*mult
        train_X_in,train_Y_in,test_X_in,test_Y = SplitData(int(tt[-1]),dataset)
        time_check = np.transpose(np.cumsum(data.fevaltime))
        fbest = []
        fbest.append(float(data.Y[0]))
        SoFarBest = []
        SoFarBest.append(int(0))
        for i in range(1,data.Y.shape[0]):
            if data.Y[i] < fbest[-1]:
                fbest.append(float(data.Y[i]))
                SoFarBest.append(int(i))
            else:
                fbest.append(fbest[-1])
        loss_tb[seed_num-1] = fbest
    Well="A"
    Well_index = 0
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_A = np.sqrt(mse_test)*max_val[Well_index]

    Well="C"
    Well_index = 1
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_B = np.sqrt(mse_test)*max_val[Well_index]

    Well="D"
    Well_index = 2
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_C = np.sqrt(mse_test)*max_val[Well_index]

    print("& ",np.round(loss_tb[seed_num-1][-1],5), "& ", np.round(sq_A,2), "& ", np.round(sq_B,2), "& ", np.round(sq_C,2))


# example data

loss_tb = np.zeros((5,50))
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_GP_'+NN_type+'_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = np.array(data.best_Y)
        tt = data.S[np.argmin(data.Y)]*mult
        train_X_in,train_Y_in,test_X_in,test_Y = SplitData(int(tt[-1]),dataset)
        time_check = np.transpose(np.cumsum(data.fevaltime))
        fbest = []
        fbest.append(float(data.Y[0]))
        SoFarBest = []
        SoFarBest.append(int(0))
        for i in range(1,data.Y.shape[0]):
            if data.Y[i] < fbest[-1]:
                fbest.append(float(data.Y[i]))
                SoFarBest.append(int(i))
            else:
                fbest.append(fbest[-1])
        loss_tb[seed_num-1] = fbest
    Well="A"
    Well_index = 0
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_A = np.sqrt(mse_test)*max_val[Well_index]

    Well="C"
    Well_index = 1
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_B = np.sqrt(mse_test)*max_val[Well_index]

    Well="D"
    Well_index = 2
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_C = np.sqrt(mse_test)*max_val[Well_index]

    
    print("& ",np.round(loss_tb[seed_num-1][-1],5), "& ", np.round(sq_A,2), "& ", np.round(sq_B,2), "& ", np.round(sq_C,2))





loss_tb = np.zeros((5,50))
for seed_num in range(1,6):
    INPUT = 'MULTI_WELL_water_butte_'+str(seed_num)
    with open('./results/RESULT_data_RBF_'+NN_type+'_'+INPUT+'.pkl', 'rb') as input:
        data = pickle.load(input)
        best_Y1 = np.array(data.best_Y)
        tt = data.S[np.argmin(data.Y)]*mult
        train_X_in,train_Y_in,test_X_in,test_Y = SplitData(int(tt[-1]),dataset)
        time_check = np.transpose(np.cumsum(data.fevaltime))
        fbest = []
        fbest.append(float(data.Y[0]))
        SoFarBest = []
        SoFarBest.append(int(0))
        for i in range(1,data.Y.shape[0]):
            if data.Y[i] < fbest[-1]:
                fbest.append(float(data.Y[i]))
                SoFarBest.append(int(i))
            else:
                fbest.append(fbest[-1])
        loss_tb[seed_num-1] = fbest
    Well="A"
    Well_index = 0
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_A = np.sqrt(mse_test)*max_val[Well_index]

    Well="C"
    Well_index = 1
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_B = np.sqrt(mse_test)*max_val[Well_index]

    Well="D"
    Well_index = 2
    mse_test = mean_squared_error(test_Y['GW_'+Well+'_0'].values, np.array(best_Y1[:,Well_index]))
    sq_C = np.sqrt(mse_test)*max_val[Well_index]

    
    print("& ",np.round(loss_tb[seed_num-1][-1],5), "& ", np.round(sq_A,2), "& ", np.round(sq_B,2), "& ", np.round(sq_C,2))



&  0.00096 &  2.4 &  2.08 &  3.71
&  0.00089 &  3.43 &  5.9 &  5.59
&  0.0007 &  2.5 &  2.24 &  3.06
&  0.00077 &  2.7 &  1.97 &  3.54
&  0.00059 &  2.13 &  1.92 &  4.43
&  0.00107 &  3.87 &  4.1 &  4.37
&  0.00082 &  2.55 &  3.85 &  4.05
&  0.00085 &  2.93 &  2.93 &  4.48
&  0.00106 &  2.3 &  2.4 &  3.79
&  0.00075 &  4.0 &  2.81 &  3.58
&  0.00091 &  2.65 &  2.55 &  4.21
&  0.0008 &  3.1 &  2.38 &  3.67
&  0.00077 &  2.08 &  1.59 &  3.88
&  0.00072 &  1.94 &  1.96 &  3.55
&  0.00081 &  4.27 &  4.09 &  4.67
